# 00 · Research protocol

> How much task-specific data and adapter capacity are required before
> parameter-efficient fine-tuning produces a measurable improvement over the
> pretrained/post-trained foundation model?

**What this notebook decides.** The estimand, the endpoints, the comparison
matrix, and — before any GPU time is spent — whether the planned evaluation is
even large enough to detect the effect worth caring about. No model is fitted
here and no data is required.

**Relationship to the docs.** [`docs/research_protocol.md`](../docs/research_protocol.md)
is normative. This notebook is its executable form: the numbers below are
computed from the same configuration files the experiments consume, so the
protocol and the code cannot drift apart silently.

In [ ]:
import pandas as pd

from qwen_text2sql.reporting import (
    environment_report,
    project_root,
)
from qwen_text2sql.reporting.style import use_house_style

ROOT = project_root()
PALETTE = use_house_style("light")
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 140)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")
ROOT

In [ ]:
report = environment_report()
print(f"commit   {report['git_commit']}")
print(f"python   {report['python']}")
print(f"platform {report['platform']}")
pd.Series(report["packages"], name="version").to_frame()

## 1 · The estimand

The primary contrast is the **paired difference in execution accuracy** between
an adapted model and its unadapted foundation-model baseline, measured on the
same examples, on databases neither model was tuned on.

"Execution accuracy" means the generated SQL is executed against the target
SQLite database and its result set is compared with the result of the verified
reference query. It is deliberately not textual similarity: two very different
queries can be equally correct, and two near-identical queries can differ in
result.

| | Endpoint | Why it is here |
|---|---|---|
| **Primary** | Execution accuracy | The only endpoint that reflects whether the query answers the question |
| Secondary | Valid SQL rate | Separates "wrote nonsense" from "wrote the wrong query" |
| Secondary | Normalized exact match | Diagnostic only — never evidence of correctness |
| Secondary | Error category | Tells you *which* capability is missing |
| Secondary | Latency | Adapters are supposed to be cheap; this checks it |

## 2 · The comparison matrix

Four conditions, each differing from a neighbour in exactly one respect, so a
difference between them is attributable.

In [ ]:
from qwen_text2sql.config import load_config

CONDITIONS = {
    "A · post-trained, unadapted": "configs/baseline.yaml",
    "B · post-trained + LoRA": "configs/qwen35_4b_lora.yaml",
    "C · post-trained + QLoRA": "configs/qwen35_4b_qlora.yaml",
    "D · base + LoRA": "configs/qwen35_4b_base_lora.yaml",
}

matrix = pd.DataFrame(
    [
        {
            "condition": label,
            "model_id": (config := load_config(ROOT / path)).model.model_id,
            "lora_rank": config.lora.rank,
            "lora_alpha": config.lora.alpha,
            "4bit": config.quantization.enabled,
            "learning_rate": config.training.learning_rate,
            "epochs": config.training.epochs,
            "seed": config.training.seed,
        }
        for label, path in CONDITIONS.items()
    ]
).set_index("condition")
matrix

Read the matrix by columns, not rows. Every column except the one under test
should be constant across the conditions being contrasted:

- **B vs A** isolates task adaptation.
- **C vs B** isolates 4-bit quantization; if `learning_rate`, `epochs` or `seed`
  differ here, the comparison is confounded and the cell above will show it.
- **D vs B** isolates general post-training, by swapping only `model_id`.

In [ ]:
varying = matrix.loc[["B · post-trained + LoRA", "C · post-trained + QLoRA"]]
differing = varying.columns[varying.nunique() > 1].tolist()
print("Columns differing between B and C:", differing or "none — the contrast is clean")

varying_bd = matrix.loc[["B · post-trained + LoRA", "D · base + LoRA"]]
differing_bd = varying_bd.columns[varying_bd.nunique() > 1].tolist()
print("Columns differing between B and D:", differing_bd or "none — the contrast is clean")

## 3 · Planned ablations

The two sweeps are generated from code rather than written down, so a cell
cannot be quietly dropped when results are aggregated later.

In [ ]:
from dataclasses import asdict

from qwen_text2sql.experiments import learning_curve_plan, rank_ablation_plan

ASSUMED_TRAIN_SIZE = 6600

plan = pd.DataFrame(
    [
        {"sweep": "learning curve", **asdict(cell)}
        for cell in learning_curve_plan(ASSUMED_TRAIN_SIZE)
    ]
    + [{"sweep": "rank ablation", **asdict(cell)} for cell in rank_ablation_plan()]
)
plan["train_size"] = plan["train_size"].astype("Int64")
plan

## 4 · Is the evaluation large enough?

This is the question that decides whether the sweep above is worth running at
all. A paired comparison only learns from examples where the two models
**disagree**, so the discordance rate matters far more than the raw evaluation
size.

Below: the smallest true difference detectable with 95% confidence and 80%
power, under McNemar's normal approximation.

In [ ]:
from qwen_text2sql.evaluation.intervals import minimum_detectable_effect

CANDIDATE_SIZES = (100, 250, 500, 1000, 2000, 5000)
DISCORDANCE_RATES = (0.10, 0.20, 0.30, 0.40)

power_table = pd.DataFrame(
    {
        f"discordance {rate:.0%}": [
            minimum_detectable_effect(n, discordance=rate) for n in CANDIDATE_SIZES
        ]
        for rate in DISCORDANCE_RATES
    },
    index=pd.Index(CANDIDATE_SIZES, name="evaluation examples"),
)
power_table.style.format("{:.1%}")

**How to read this.** Pick the row matching your validation split and the column
matching how differently the two models behave. The cell is the smallest true
improvement you could reliably detect.

If that number is larger than the improvement you expect, the experiment is not
underpowered *in principle* — it simply cannot answer the question, and running
it produces a confident-looking result that is mostly noise. The fix is a larger
evaluation set, not a larger model.

Two honest caveats:

- This is a planning approximation. The paired bootstrap in
  [`05_execution_evaluation`](05_execution_evaluation.ipynb) is the actual
  inference procedure.
- BIRD examples cluster by database. Examples within one schema are not
  independent, so the effective sample size is nearer the number of *databases*
  than the number of questions, and these figures are optimistic.

## 5 · Rules this project holds itself to

1. Never optimise a decoding or training choice on the final evaluation set.
2. Keep schema-identical databases out of both sides of a train/validation split.
3. Save configuration, dataset hash, model ID and training-set size with every run.
4. Keep per-example predictions and evaluation records, not only aggregate scores.
5. Compare models on the same examples, and report uncertainty for differences.
6. Do not claim semantic correctness from SQL text similarity alone.
7. Treat benchmark annotation errors separately from model errors.

Rule 2 is verified mechanically in [`01_data_audit`](01_data_audit.ipynb).
Rule 5 is enforced by the comparison helpers, which refuse to pair runs scored
on different example sets.

## What would falsify the hypothesis

Fine-tuning does **not** help if held-out execution accuracy fails to improve
beyond sampling uncertainty, if gains vanish on unseen schemas, or if gains are
explained by evaluation leakage. Each of those is checked later in the sequence
rather than assumed away.

---

### Provenance

Every number above is produced by code in `src/qwen_text2sql/`, which is unit
tested and type checked. This notebook orchestrates and interprets; it does not
define the analysis. If a figure here looks wrong, the fix belongs in `src/` and
comes with a test.